# 1.2 Sequential API

這份 Notebook 使用 `tf.keras.Sequential` 建立二元分類模型。Sequential API 適合一層接一層的單一路徑模型，是建立 baseline 的首選。


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

tf.keras.utils.set_random_seed(42)


## 1. 建立非線性二元分類資料

`make_moons` 是二維資料，但類別邊界不是直線。這很適合拿來示範 DNN 如何學習非線性分類。


In [ ]:
X, y = make_moons(n_samples=1000, noise=0.22, random_state=42)

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', s=18)
plt.title('make_moons dataset')
plt.xlabel('feature 1')
plt.ylabel('feature 2')
plt.show()


## 2. 切分資料與標準化


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype('float32')
x_test = scaler.transform(x_test).astype('float32')
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')

print('x_train shape:', x_train.shape)
print('x_test shape:', x_test.shape)


## 3. 使用 Sequential API 建立模型


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


## 4. 訓練模型


In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=60,
    batch_size=32,
    verbose=0
)


## 5. 評估模型


In [ ]:
train_result = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
test_result = model.evaluate(x_test, y_test, verbose=0, return_dict=True)

pd.DataFrame([train_result, test_result], index=['train', 'test'])


In [ ]:
y_prob = model.predict(x_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification report:')
print(classification_report(y_test, y_pred, digits=4))


## 6. 觀察訓練曲線


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Sequential API Training Curve')
plt.show()


## 7. 小結

Sequential API 的重點是把 layer 依序放進 `tf.keras.Sequential([...])`。只要模型是單一路徑，這種寫法最清楚也最容易維護。
